# 002: pixel-by-pixel phase mapping

Written by Jean-Baptiste Jacob

Last updated: 18/02/2025

Map peaks on a 2D pixel grid and find the best-matching phase for each pixel among a list of pre-defined crystal structures. 

This approach assumes that each pixel on the map contains one dominant mineral, excluding tiny inclusions, secondary phases etc. If you goal is to look at these tiny grains and how they are distributed in the sample, this is probably not the right method. You can use this notebook to map the major phases, and then save the residual peaks (not assigned to any major phase) and process it separately. 

### Load packages

In [ ]:
import sys

# python environment stuff
IMAGED11_PATH = '/home/esrf/jean1994b/ImageD11_jbjacob'  # None means do not use git, otherwise enter the name of the folder to use for the git checkout "ImageD11" or "ImageD11_version_xx", etc
CHECKOUT_PATH = 'ImageD11'  # the name of the git checkout folder within path. None means guess


if IMAGED11_PATH is not None:
    if '/data/id11/nanoscope' not in sys.path:
        sys.path.append('/data/id11/nanoscope')
    import install_ImageD11_from_git
    install_ImageD11_from_git.setup_ImageD11_from_git(IMAGED11_PATH,CHECKOUT_PATH)

In [ ]:
# general modules
import os, glob
import h5py
import matplotlib.pyplot as plt
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
from tqdm import tqdm

# ImageD11 https://github.com/FABLE-3DXRD/ImageD11
import ImageD11.sinograms.dataset
import ImageD11.columnfile
import ImageD11.parameters
import ImageD11.friedel_pairs as fp

# point-fit 3dxrd module available at https://github.com/jbjacob94/pf_3dxrd.
# Not yet installable with pip, just copy + paste the files into your working folder   
from pf3dxrd.pf3dxrd import utils, pixelmap, crystal_structure, peak_mapping, phase_mapping

%load_ext autoreload
%autoreload 2
%matplotlib ipympl

### Load data & crystal structures


In [ ]:
dsfile = 'MgO_3_0p1M_12um_0003/MgO_3_0p1M_12um_0003_dataset.h5'

In [ ]:
ds = ImageD11.sinograms.dataset.load(dsfile)
print(ds)

ds.y0 = 11.230555555555555  # copy the fitted value from fit_y0 notebook

is_subset=False             # default. this is set to True two cells below if you want to work with a subset instead of the full peakfile, for tuning soem parameters

In [ ]:
col2dfile = ds.col2dfile.replace('.h5','_paired.h5')
cf = ImageD11.columnfile.colfile_from_hdf(col2dfile)
cf.parameters.loadparameters(ds.parfile)

fp.update_geometry_fpairs(cf,ds, relocate_pairs=True)
print(cf.nrows)

#### (optional): take a subset of cf
useful to tune params when cf is large. saved as separate file to be used for preliminary indxing (allows to better refine cell pars

In [ ]:
# take a subset of cf (useful for tuning params in phase selection mask)
"""i1, i2 = fp.get_pairs(cf, 'omega')
rnd = np.random.choice([True,False], len(i1), p=(0.1,0.9))

i1s, i2s = i1[rnd], i2[rnd]

to_keep = np.full(cf.nrows, False)
to_keep[i1s] = True
to_keep[i2s] = True

cf.filter(to_keep)
is_subset = True"""

In [ ]:
# plot sample reconstruction to check it is ok
kw = {'cmap':'Greys_r', 'vmin':0.01, 'vmax':1}
fig, _ = utils.friedel_recon(cf, ds.ybinedges-ds.y0, ds.ybinedges-ds.y0, doplot=True, norm=True, mask = None, **kw)

In [ ]:
# save the image if you want
fig.savefig(ds.dsfile.replace('dataset.h5','friedel_recon.png'), format='png', dpi=300)

In [ ]:
# filter peaks relocated outside the box [y0-ymax,y0-ymax,y0+ymax,y0+ymax]
This step is important 
mask = ( np.abs(cf.sx) <= ds.ymax-ds.y0) & ( np.abs(cf.sy) <= ds.ymax-ds.y0)
cf.filter(mask)
cf.nrows

pf3dxrd uses a custom `crystal_structure` (CS) class to store crystallographic information. Each CS instance is created from a cif file, and is built on other Python packages for X-ray diffraction data, namely `diffpy.structure`, `Dans_Diffraction`, and `orix`. I recommend creating a cif folder to put all the potential crystal structure sin the sample, and then load them with the cell below

In [ ]:
# load crystal structures and store them in a dictionnary
pnames = ['MgO_cubic'] # cif files should be named '<phase_name>.cif'

phase_dict = {name.split('_')[0]: crystal_structure.load_CS_from_cif(cif_path = f'../cif/{name}.cif', name=name.split('_')[0], pid = i) for i,name in enumerate(pnames)}

for cs in phase_dict.values():
    print(cs)


### Initialize 2D map (pixelmap) and PhaseMapper object

Phase mapping is performed on a 2D grid of pixels, where the size of the grid is determined by the scanning range along the y-axis and the resolution is given by the y-step size. To make this process more convenient, a specific class, `Pixelmap`, is introduced. It enables storing properties (phase ids, grain_ids, U, nb of peaks, etc.) onto a fixed 2D grid, alongside with crystal structure information for each phase in the map. It is very similar to `Tensormap` in ImageD11, but has additional functionalities for grain mapping and plotting. From now on, it will be the main object we work with, alongside `ImageD11` columniles.  

**NOTE**: a pixelmap (xmap) instance can be easily converted to Tensormap to get a consistent final format across all s3dxrd pipelines.

The `PhaseMapper` handles the phase mapping process, i.e. assigning a phase to each pixel in xmap. 


In [ ]:
# pixelmap: object to store and plot data on a 2D grid. Initialize it from dataset
xmap = pixelmap.create_from_dataset(ds, h5name = os.path.join(ds.analysispath, ds.dsname+'_xmap.h5') )
    
#pixelmap phases: add them from dict
for phasename, cs in phase_dict.items():
    xmap.phases.add_phase(phasename, cs)
    
print(xmap)

In [ ]:
# PhaseMapper object to map peaks to pixel and find best phase match on the 2D grid
PhaseMapper = phase_mapping.PhaseMapper(cf, ds, xmap, CS_list=phase_dict.values())

print('================= \n',PhaseMapper)

### Compute raw phase masks
Compute selection mask for each phases in PhaseMapper. Peak selection in each mask is based on a simple 2-theta threshold selection around pre-computed theoretical 2-theta positions of Bragg peaks. 
A peak can therefore belong to multiple phases masks when there is overlap. The mapping process consists in resolving locally these conflicts by assigning to each pixel the dominant phase, that maximizes a completeness criterion. 


#### Raw phase masks
`find_strongest_bragg_peaks` simulates the powder diffraction pattern for a given crystal structure over a given 2-theta range. A peak search is then performed to find the N-strongest peaks. 
`Imin` and `prominence`, and `Nmax` control how many peaks to keep and what is considered a peak. 

In [ ]:
# first identify position of Bragg peaks for all phases
PhaseMapper.find_strongest_bragg_peaks_all(0, cf.tth.max(), Nmax=90, Imin=0.001, prominence=0.001, doplot=True) 

Compute phase masks. The proportion of peaks selected in each mask is returned, as well as the total fraction of peaks assigned (selected by at least one phase) and the fraction of peak overlaps (fraction of peaks assigned to more than one phase).

In [ ]:
# compute phase masks
PhaseMapper.compute_phase_mask_all(tth_max=cf.tth.max(), tth_tol=0.022)

In [ ]:
# plot tth vs eta for a subset of peaks to see what the selection looks like
PhaseMapper.plot_tth_eta(min_tth=3, max_tth=16.5, show_theorytth=True, phase_colors='from_mask')

In [ ]:
# can also look at peaks histogram in d-star
PhaseMapper.plot_ds_histogram(minval=0, maxval=1.6, step_size = 0.0001, show_theoryds = True, mask=None)

What you want here is predicted peaks from different phases that match the observed peaks in the histogram. For simple cases (only one phase, high symmetry; e.g. pure Al) it is straightforward, but fore more complex samples for which you don't know exaclty what's in there this may require a bit of fiddling around, playing whith the different phases to integrate or not in the map. Another complication often arises from poorly fitted unit cell parameters. In this case, it can be useful to take a subset (e.g. a small pixel domain in one grain), index it (see indexing notebook), write a refined cif file with the fitted unit cell and then reload it in xmap. 

### Phase labeling

In [ ]:
# run this function to set up PhaseMapper for phase labelling. This initializes some variables and make sure the peakfile is sorted properly. 
PhaseMapper.get_ready_for_phase_labeling()

In [ ]:
# plot the distribution of peak counts per pixel, to adjust the minpks parameter (below).
def npks_per_pixel_hist(cf):
    px, cnts = np.unique(cf.xyi, return_counts=True)
    plt.figure()
    plt.hist(cnts[cnts>0],100);
    plt.xlabel('pks per pixel')
    plt.ylabel('N pixel')
    
npks_per_pixel_hist(cf)
    


Set up parameters in PhaseMapper
- minpks : minimum number of peaks in a pixel, below which it will remained unlabeled
- min_confidence : lower threshold for normalized confidenc eindex, below which the pixel will remain unlabeled
- kernel_size : peak selection is done in a $n \times n$ kernel around each pixel. default is n=1 (signel pixel selection), but kernel size can be increased to get smoother map. If larger kernel size is used, think of increasing minpks accordingly
- chunksize : size of chunks for multiprocessing
- ncpu : number of cpus used for multiprocessing. Default is the largest number available
- res: dictionnary in which phase labelign outputs are temporarily written

In [ ]:
PhaseMapper.kernel_size = 3  # kernel size=3 is usually a good compromise to get smooth maps that do not blur too much the details at phase boundaries
PhaseMapper.minpks = 600
PhaseMapper.min_confidence = 0.6   # min confidence index. this value can vary a lot depending on how many phases are considered. May be high (>0.7) for maps with 1-2 phases, but can decrease significantly (<0.2) whan adding many phases 
PhaseMapper.chunksize = 800
PhaseMapper.ncpu
PhaseMapper.res = {}

In [ ]:
# run phase labeling (parallelized). results are stored in a dictionnary attribute in PhaseMapper
PhaseMapper.label_phase(parallelize=True)

In [ ]:
# write outputs in pixelmap. peakfile phase_ids column is also updated at the same time, giving a phase label to each peak. non-labeled peaks set at -1
PhaseMapper.results_to_xmap()

In [ ]:
# print some stats about laleled peaks: fraction of peaks assigned to each phase (in proportion of the total number of peaks in peakfile)
PhaseMapper.get_stats_labeled_peaks()

Additional variables added to xmap:
- `phase_map_completeness`: how much intensity is matched by the selected phase, compared to total intensity over that pixel. fraction between 0 and 1
- `phase_map_uniqueness`  : quantifies the degree of peaks overlap among different phases in each pixels. takes a value between 0 and 1, 1 meaning no overlap.
- `phase_label_confidence` :criterion between 0 and 1 built from the two above
- `Npks` : number of peaks matched in each pixel
- `phase_ids` : Label identifying each phase. It corresponds to the `phase_id` defined in `xmap.phases.[phasename].phase_id`.

In [ ]:
# make some plots and save
plt.close('all')

colors = ((0,0,0),) + plt.matplotlib.cm.tab10.colors
cmap = plt.matplotlib.colors.ListedColormap( colors[:len(xmap.phases.pnames)] )

save=False
var_to_plot = ['phase_map_completeness',
               'phase_map_uniqueness',
               'phase_label_confidence',
               'Npks',
               'phase_ids']


for i, var in enumerate(var_to_plot):
    if var == 'phase_id':
        kw = {'cmap':cmap}
        xmap.plot(var, autoscale=False, save=save, **kw)
        
    else:
        kw = {'cmap':'viridis'}
        xmap.plot(var, autoscale=False, hist_tails_cut=[1, 95], save=save, **kw)


### Filter peak file and save outputs

In [ ]:
# sanity check: make sure phase_ids labels have been updated in peakfile 
np.unique(cf.phase_ids, return_counts=True)

In [ ]:
# remove phase mask columns in peakfile. we don't need them anymore
columns_to_drop = PhaseMapper.phases.pnames+['overlap','assigned']

cf = ImageD11.columnfile.colfile_from_dict({t:cf.getcolumn(t) for t in cf.titles if t not in columns_to_drop})

In [ ]:
# filter out unlabeled peaks (phase_id = -1) ; keep them in parallel colfile (residuals)
labeled = cf.phase_ids != -1  # unlabeled peaks. may contain peaks in phase masks but located in rejected pixels

notassigned = ~PhaseMapper.peakfile.assigned  # peaks not in any phase mask
cf_residuals = ImageD11.columnfile.colfile_from_dict({t:cf.getcolumn(t)[notassigned] for t in cf.titles})
cf_residuals.nrows

In [ ]:
cf.filter(labeled)

In [ ]:
# sanity check: make sure filtering has not splitted up some friedel pairs. In such case, you would not be able to recompute friedel pair geometric corrections 
# when reloading the peakfile (friedel_pairs.update_geometry_s3dxrd).
i1, i2 = fp.get_pairs(cf, 'omega')
np.all(np.equal(cf.omega_pair_id[i1], cf.omega_pair_id[i2]))

If everything is allright, save labeled peakfile and pixelmap

In [ ]:
# save new cf
if is_subset:
    outname = ds.col2dfile.replace('.h5','_subset.h5')  # use different name if saving a subset, to avoid overwriting the main paired peaks file
else:
    outname = ds.col2dfile.replace('.h5','_paired.h5')  # overwrite paired peaks file
    
utils.colf_to_hdf(cf, outname, save_mode = 'minimal')

# save pixelmap
xmap.save_to_hdf5()

### Have a look at unlabeled peaks

In [ ]:
fig, _ = utils.friedel_recon(cf_residuals, ds.ybinedges-ds.y0, ds.ybinedges-ds.y0, doplot=True, norm=True, **kw)

In [ ]:
cf_residuals.parameters.loadparameters(ds.parfile)
PhaseMapper.peakfile = cf_residuals

In [ ]:
PhaseMapper.plot_ds_histogram(0, 1.6, 0.0002, show_theoryds=True)

In [ ]:
# save them as separate file
outname = ds.col2dfile.replace('.h5','_unlabeled.h5')
utils.colf_to_hdf(cf_residuals, outname, save_mode = 'minimal')